# Devoir 5 - Solaire et applications
### Ève Castonguay & Tristan Tessier 

## **Question 3 - Dimensionnement d'un système PV et thermique**

In [2]:
# Imports
using Pkg
Pkg.activate(".")
Pkg.add("LaTeXStrings")   
Pkg.instantiate()
using DataFrames  
using Dates  
using CSV  
using Statistics  
using PlotlyJS 
using Unitful
using Printf
using LaTeXStrings 
using Glob # * dans les str

  Activating project at `~/Desktop/Solaire/D5`
     Project No packages added to or removed from `~/Desktop/Solaire/D5/Project.toml`
    Manifest No packages added to or removed from `~/Desktop/Solaire/D5/Manifest.toml`


In [49]:
# Folder names
path_name = "/Users/evecastonguay/Library/CloudStorage/OneDrive-polymtl/Aut. 2026 - Polytechnique/Solaire/D5/"

# File names
integrals = "*-Integrals.csv"
checks = "*-Checks.csv"
thermal = "*-Thermal.csv"
pv = "*-PvDetails.csv"

file_list = [integrals,checks,thermal,pv]

4-element Vector{String}:
 "*-Integrals.csv"
 "*-Checks.csv"
 "*-Thermal.csv"
 "*-PvDetails.csv"

In [50]:
# Remove first December month
function remove_december(df)
    start_hour = 8760
    end_hour = 17520
    df_year = df[start_hour .<= df[:, 1] .<= end_hour, :]
    return df_year
end

remove_december (generic function with 1 method)

## Résultats pour une simu donnée

### Valeur integrals

In [56]:
## Integral
folder = "depart"
data_folder = path_name * folder
type_file = 1 

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame, stripwhitespace = true) # glob(pattern,dossier) returns array
rename!(df, strip.(names(df))) # remove spaces after column names
#print(df)
stats = DataFrame(
    PPv = df.PPv .* u"kJ",
    PGrid = df.PGrid .* u"kJ",
    PDumped = df.PDumped .* u"kJ",
    PAuxHeat = df.PAuxHeat .* u"kJ",
    PAuxDhw = df.PAuxDhw .* u"kJ",
    PPump = df.PPump .* u"kJ",
    PTot = df.PTot .* u"kJ",
    EincPv = df.EincPv .* u"kJ",
    fSolElectrical = df.fSolElectrical,
    η_col = df.etaCol,
    fSolThermal = df.fSolThermal
)
# Rendement systeme
η_sys = (stats.PTot[1] - stats.PGrid[1]) / stats.EincPv[1]
# Validation dimensionnement
pourcentage_pgrid = (stats.PGrid[1] / stats.PTot[1]) * 100

@printf("\nValidation dim. #1 : f. sol. élec. est %.6f", stats.fSolElectrical[1])
@printf("\nValidation dim. #2: %.6f", pourcentage_pgrid)
@printf("\nLe rendement du système est %.2f", η_sys)
@printf("\nLe rendement des capteurs est %.2f", stats.η_col[1])
@printf("\nLa fraction solaire thermale des capteurs est %.2f", stats.fSolThermal[1])
@printf("\nPGrid = %.2f", stats.PGrid[1])
@printf("\nPDumped = %.2f", stats.PDumped[1])
@printf("\nDécomposition PTot : \n PTot = %.2f,\n Ppump = %.2f,\n PAuxHeat = %.2f,\n PAuxDhw = %.2f", stats.PTot[1], stats.PPump[1], stats.PAuxHeat[1], stats.PAuxDhw[1])



Validation dim. #1 : f. sol. élec. est 0.421002
Validation dim. #2: 57.899782
Le rendement du système est 0.08
Le rendement des capteurs est 0.35
La fraction solaire thermale des capteurs est 0.37
PGrid = 23556834.91 kJ
PDumped = 10516601.10 kJ
Décomposition PTot : 
 PTot = 40685532.88 kJ,
 Ppump = 1002384.00 kJ,
 PAuxHeat = 21615913.88 kJ,
 PAuxDhw = 2561477.38 kJ

### Histogramme PGrid

In [47]:
## PV
type_file = 4 # PV

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame) # glob(pattern,dossier) returns array
df = remove_december(df)

# Mois de l'année : heure de fin de chaque mois (année non bissextile)
mois = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heures_fin_mois = cumsum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]) .* 24

# 3.1 - Get data from file
rename!(df, strip.(names(df))) # remove spaces after column names
stats = DataFrame(
    time = df.TIME .* u"hr",
    PPv = df.PPv .* u"kJ/hr",
    PGrid = df.PGrid .* u"kJ/hr",
    PTot = df.PTot .* u"kJ/hr",
    TPvCell = df.TPvCell .* u"°C"
)
# temps
Δt = stats.time .- vcat(stats.time[1], stats.time[1:end-1])
stats.Δt = Δt
# Fraction solaire electrique
Etot = stats.PTot .* Δt
Egrid = stats.PGrid .* Δt
Fe = (Etot.-Egrid) ./Etot
stats.Fe = Fe
stats.Egrid = Egrid 
# Fraction solaire mensuelle
h = ustrip.(u"hr", stats.time) .- 8760               # heure de l'année (0 → 8760)
m = searchsortedfirst.(Ref(heures_fin_mois), h)      # n° du mois de chaque pas de temps
Fe_mois = [(sum(Etot[m .== k]) - sum(Egrid[m .== k])) / sum(Etot[m .== k]) for k in 1:12]
# PGrid
Egrid_mois = [ustrip(u"GJ", sum(Egrid[m .== k])) for k in 1:12]   # énergie du réseau par mois [GJ]

## Section 4 - Graphique : énergie mensuelle tirée du réseau
trace_Egrid = bar(x = mois, y = Egrid_mois,
                  marker = attr(color = "#636EFA", opacity = 0.6,
                                line = attr(color = "black", width = 1)))
fig_Egrid = plot(trace_Egrid,
                 Layout(width = 800,
                    height = 400,
                    xaxis_title = "Mois",
                    yaxis_title = "Énergie du réseau [GJ]",
                    showlegend = false))
savefig(fig_Egrid, folder * "-energie_reseau_mensuelle.png", scale = 5)

"depart-energie_reseau_mensuelle.png"

### Histo PTot et PPv

In [48]:
## PV
type_file = 4 # PV

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame) # glob(pattern,dossier) returns array
df = remove_december(df)

# Mois de l'année : heure de fin de chaque mois (année non bissextile)
mois = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heures_fin_mois = cumsum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]) .* 24

# 3.1 - Get data from file
rename!(df, strip.(names(df))) # remove spaces after column names
stats = DataFrame(
    time = df.TIME .* u"hr",
    PPv = df.PPv .* u"kJ/hr",
    PGrid = df.PGrid .* u"kJ/hr",
    PTot = df.PTot .* u"kJ/hr",
    TPvCell = df.TPvCell .* u"°C"
)
# temps
Δt = stats.time .- vcat(stats.time[1], stats.time[1:end-1])
stats.Δt = Δt
# Fraction solaire electrique
Etot = stats.PTot .* Δt
Egrid = stats.PGrid .* Δt
Fe = (Etot.-Egrid) ./Etot
stats.Fe = Fe
stats.Egrid = Egrid 
# Fraction solaire mensuelle
h = ustrip.(u"hr", stats.time) .- 8760               # heure de l'année (0 → 8760)
m = searchsortedfirst.(Ref(heures_fin_mois), h)      # n° du mois de chaque pas de temps
Fe_mois = [(sum(Etot[m .== k]) - sum(Egrid[m .== k])) / sum(Etot[m .== k]) for k in 1:12]
# PGrid
Egrid_mois = [ustrip(u"GJ", sum(Egrid[m .== k])) for k in 1:12]   # énergie du réseau par mois [GJ]
# PTot
ETot_mois = [ustrip(u"GJ", sum(Etot[m .== k])) for k in 1:12] 
# PPv
EPv = stats.PPv .* Δt
EPv_mois = [ustrip(u"GJ", sum(EPv[m .== k])) for k in 1:12] 

## Section 4 - Graphique : énergie mensuelle tirée du réseau
trace1 = bar(x = mois, y = ETot_mois,
                    name = "Besoins totaux",
                  marker = attr(color = "#636EFA", opacity = 0.6, 
                                line = attr(color = "black", width = 1)))
trace2 = bar(x = mois, y = EPv_mois,
                    name = "Production PV",
                  marker = attr(color = "#FF6692", opacity = 0.6,
                                line = attr(color = "black", width = 1)))
fig = plot([trace1,trace2],
                 Layout(width = 800,
                    height = 400,
                    xaxis_title = "Mois",
                    yaxis_title = "Énergie [GJ]",
                    showlegend = true,
                    legend = attr(x = 0.5, y = 0.99, xanchor = "right", yanchor = "top")))
savefig(fig, folder * "-PTot-PPv.png", scale = 5)

"depart-PTot-PPv.png"

## Dimensionnement

#### Fraction solaire et autres données integrals

In [60]:
# Nom de la simulation
folder = "run2"

"run2"

In [61]:
## Integral
data_folder = path_name * folder
type_file = 1 

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame, stripwhitespace = true) # glob(pattern,dossier) returns array
rename!(df, strip.(names(df))) # remove spaces after column names
#print(df)
stats = DataFrame(
    PPv = df.PPv .* u"kJ",
    PGrid = df.PGrid .* u"kJ",
    PDumped = df.PDumped .* u"kJ",
    PAuxHeat = df.PAuxHeat .* u"kJ",
    PAuxDhw = df.PAuxDhw .* u"kJ",
    PPump = df.PPump .* u"kJ",
    PTot = df.PTot .* u"kJ",
    EincPv = df.EincPv .* u"kJ",
    fSolElectrical = df.fSolElectrical,
    η_col = df.etaCol,
    fSolThermal = df.fSolThermal
)
# Rendement systeme
η_sys = (stats.PTot[1] - stats.PGrid[1]) / stats.EincPv[1]
# Validation dimensionnement
pourcentage_pgrid = (stats.PGrid[1] / stats.PTot[1]) * 100

@printf("\nValidation dim. #1 : f. sol. élec. est %.6f", stats.fSolElectrical[1])
@printf("\nValidation dim. #2: %.6f", pourcentage_pgrid)
@printf("\nLe rendement du système est %.2f", η_sys)
@printf("\nLe rendement des capteurs est %.2f", stats.η_col[1])
@printf("\nLa fraction solaire thermale des capteurs est %.2f", stats.fSolThermal[1])
@printf("\nPGrid = %.2f", stats.PGrid[1])
@printf("\nPDumped = %.2f", stats.PDumped[1])
@printf("\nDécomposition PTot : \n PTot = %.2f,\n Ppump = %.2f,\n PAuxHeat = %.2f,\n PAuxDhw = %.2f", stats.PTot[1], stats.PPump[1], stats.PAuxHeat[1], stats.PAuxDhw[1])



Validation dim. #1 : f. sol. élec. est 0.998091
Validation dim. #2: 0.190875
Le rendement du système est 0.03
Le rendement des capteurs est 0.30
La fraction solaire thermale des capteurs est 0.47
PGrid = 70748.07 kJ
PDumped = 117182815.17 kJ
Décomposition PTot : 
 PTot = 37065216.86 kJ,
 Ppump = 1295112.96 kJ,
 PAuxHeat = 18490107.95 kJ,
 PAuxDhw = 1774238.33 kJ

#### Prochain dimensionnement

In [62]:
# Pour le calcul des coûts
cout_Ag = 1000 # $/m2
cout_Vs = 6000 # $/m3
cout_CBat = 750 # $/kWh
cout_PPeak = 2 # $/Wp

########### 
# Section 1: dimensionner PPeak sur mois critique 
###########

## Étape 1: valeur PPeak dans Integrals 
data_folder = path_name * folder
type_file = 1 
df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame, stripwhitespace = true) # glob(pattern,dossier) returns array
rename!(df, strip.(names(df)))

PPeak = df.PPeakPv[1] # kWp
Ag = df.Ag[1] # m2
Vs = df.Vs[1] # L/m2
CBat = df.CBat[1] # kWh
pente = df.slope[1] #°

## Étape 2 : integrale PPv
type_file = 4 # PV
df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame) # glob(pattern,dossier) returns array
df = remove_december(df)

# Mois de l'année : heure de fin de chaque mois (année non bissextile)
mois = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heures_fin_mois = cumsum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]) .* 24

rename!(df, strip.(names(df))) # remove spaces after column names
stats = DataFrame(
    time = df.TIME .* u"hr",
    PPv = df.PPv .* u"kJ/hr",
    PGrid = df.PGrid .* u"kJ/hr",
    PTot = df.PTot .* u"kJ/hr",
    TPvCell = df.TPvCell .* u"°C",
    PDumped = df.PDumped .* u"kJ/hr",
    PLoad = df.PLoad .* u"kJ/hr"
)
# pour intégrales sur mois
h = ustrip.(u"hr", stats.time) .- 8760               # heure de l'année (0 → 8760)
m = searchsortedfirst.(Ref(heures_fin_mois), h)      # n° du mois de chaque pas de temps
# temps
Δt = stats.time .- vcat(stats.time[1], stats.time[1:end-1])
stats.Δt = Δt
# η_ond (par pas de temps)
η_ond_heure = (stats.PLoad.-stats.PGrid) ./ (stats.PPv.-stats.PDumped)

# Fonction -- energie mensuelle à partir d'une puissance (kJ/hr) → vecteur de 12 valeurs [kWh par défaut]
function energie_mois(P, Δt, m; unite = u"kW*hr")
    E = P .* Δt # kJ
    return [ustrip(unite, sum(E[m .== k])) for k in 1:12]
end

ppv = stats.PPv .* Δt # kJ 
ppv_mois_test = [ustrip(u"kW*hr", sum(ppv[m .== k])) for k in 1:12] # [kWh]
PPv_mois = energie_mois(stats.PPv,stats.Δt,m) # --> PPv [kWh]
if ppv_mois_test != PPv_mois
    print("ERREUR DANS CALCUL DES MOIS")
end

ETot_mois = energie_mois(stats.PTot, Δt, m)    
PDumped_mois = energie_mois(stats.PDumped, Δt, m)                # [kWh]
PPv_mois     = energie_mois(stats.PPv, Δt, m)                    # [kWh]
PGrid_mois   = energie_mois(stats.PGrid, Δt, m)  
PLoad_mois   = energie_mois(stats.PLoad, Δt, m)  

## Étape 3 : calculer y
y_mois = PPv_mois ./ PPeak # [kWh/kWp] combien de kWh sont produits par kWp de PV?

## Étape 4 : trouver rendement onduleur
η_ond_mois = (PLoad_mois.-PGrid_mois) ./ (PPv_mois.-PDumped_mois)

## Étape 5 : dimensionnement de PPeak (trouver mois critique)
PPeak_mois = ETot_mois ./ (y_mois .* η_ond_mois)
PPeak_min, i_crit = findmax(PPeak_mois)           # valeur max et son indice
mois_critique = mois[i_crit]  
@printf("Le mois critique pour la simulation %s est %s\n", folder, mois_critique)
@printf("Valeur de PPmax associée est %.2f\n", PPeak_min)

########### 
# Section 2: dimensionner CBat (sans rouler plusieurs simulations avec différents PPeak)
###########

# Étape 1 : pour le fun, trouver la plus grande décharge dans la simulation 
# Ça donne rien , peut-être que batterie était pas assez grosse et la plus grande est 0!
# Expliquer concept dans rapport

# Étape 2 : production PPv mise à l'échelle pour autre valeur de PPeak
# Hypothèse: si on augmente le nombre de panneaux PV, leur production sera proportionnelle

# Fonction pour calculer le plus grand déficit pour le vecteur nt des ppv
function capacite_utile(n; dt = 0.25)
    # n : vecteur du bilan net n(t) en kJ/h (surplus > 0, déficit < 0)
    D = 0.0          # déficit cumulé courant (kJ)
    Dmax = 0.0       # plus grand déficit rencontré (kJ)
    for k in 1:2                      # 2 passages sur l'année
        for nt in n
            D = max(0.0, D - nt * dt) # kJ
            if k == 2                 # on ne garde que le 2e passage
                Dmax = max(Dmax, D)
            end
        end
    end
    return Dmax / 3600   # kJ --> kWh
end

# Étape 2.1 : trouver une série de valeurs PPMax
# reprendre valeur PPeak_min et trouver multiple de .25 le plus proche
multiple = 0.25
PPeak_max_mois_critique = round(PPeak_min / multiple) * multiple
# trouver une série de valeurs plus grandes et tester
valeurs_hypothetiques_ppeak = PPeak_max_mois_critique .+ (0:20) .* 0.5   # PPmax, PPmax+5, …, PPmax+20 [kWp]
@printf("Les extrêmes sont %.2f et %.2f",valeurs_hypothetiques_ppeak[1],valeurs_hypothetiques_ppeak[end])

# Étape 2.2 : boucle sur ces nombres
vecteur_CBat = zeros(length(valeurs_hypothetiques_ppeak)) # kWh
vecteur_couts = zeros(length(valeurs_hypothetiques_ppeak))
for (i,val_ppeak) in enumerate(valeurs_hypothetiques_ppeak)
    # n(t) = η · (PPeak / PPeak_sim) · P_Pv(t) − P_Tot(t)
    ppv_hypothetique = (stats.PPv ./ PPeak) .* val_ppeak # [kJ/h] pour chq pas de temps

    # Étape 3 : dimensionnement CBat
    η = η_ond_mois[i_crit]
    puissance_batterie = (η .* ppv_hypothetique) .- stats.PTot # - : décharge batterie. + : charge batterie [kJ/hr]
    # Calcul du plus grand déficit cumulé 
    # D(t) = max(0, D(t−1) − n(t) · 0.25)
    CBat_max = capacite_utile(ustrip.(u"kJ/hr", puissance_batterie))/0.9 # CBat --> kWh et SOC min est 0.1 dans TRNSYS (1-0.1=0.9)
    # ajouter au vecteur
    vecteur_CBat[i] = CBat_max

    # Étape 4 : calcul coût
    cout_total = (Ag*cout_Ag) + (Ag * Vs * 0.001 * cout_Vs) + (CBat_max*cout_CBat) + (val_ppeak*1000*cout_PPeak)
    # ajouter au vecteur
    vecteur_couts[i] = cout_total
end

# Étape 5 : trouver le coût minimum
min_val, min_idx = findmin(vecteur_couts)
# PPeak idéal
PPeak_ideal = valeurs_hypothetiques_ppeak[min_idx]
CBat_ideal = vecteur_CBat[min_idx]
@printf("\nLa combinaison idéale pour \nAg= %.2f  \nVs = %.2f  \nest: \nPPeak=%.2f  \nCBat=%.2f  ",Ag,Vs,PPeak_ideal,CBat_ideal)
print("\nMaintenant, réessayer avec un autre couple Ag et Vs.")

Le mois critique pour la simulation run2 est Dec
Valeur de PPmax associée est 24.36
Les extrêmes sont 24.25 et 34.25
La combinaison idéale pour 
Ag= 15.20  
Vs = 22.00  
est: 
PPeak=27.75  
CBat=149.91  
Maintenant, réessayer avec un autre couple Ag et Vs.

In [ ]:
## dans PV
#### Section 1: dimensionner PPeak (valeur minimale, mais force batterie à couvrir série jours nuageux ou déficit reporté du mois précédent)
# 1) calculer PPv
# P_Pv [kJ/h] × 0.25 [h] = énergie du pas en kJ
# Σ (P_Pv × 0.25) donne l’énergie du mois, toujours en kJ.
# kJ / 3600 = kWh.

# 2) calculer y
# y = E_Pv,mois [kWh] / PPeakPv [kW], donc y est en kWh/kW

# 3) calculer rendement onduleur mois critique (decembre)
# P_Pv(n’importe quelle puissance) ≈ y × PPeak -> Ptot / (y*η) = PPeak nécessaire pour subvenir à ce mois (on a dimensionné PPeak)

#### Section 2: (pour un PTot donné) dimensionner CBat pour plusieurs valeurs de PPeak (garder couple le moins cher)
# 1) faire augmenter PPeak


# z) noter coût total (thermique + réservoir + PV + batterie)

#### Section 3: Recommencer 1-2 avec autre couple (Ac, Vs) 
# 1. comparer prix. si cout baisse, garder nouvelle config.

#### Section 4: jouer avec la pente
